# Model A localisation: standalone Bernoulli analysis

## Goal and run instructions

This notebook takes **Model A only** from `localisation_analysis.ipynb`. Each forecast is scored
as a binary hit or miss. The statistical model is hierarchical Bernoulli regression with a
logit link, the centred configuration effects and zero-sum harmonic/background effects of that
source. There is no residual Student-t scale. Student-t distributions remain as parameter priors.

1. In Colab, a GPU runtime is useful for **Chronos collection**. The Bayesian fit runs on CPU
   with nutpie when installed, otherwise PyMC. It can also run entirely on CPU.
2. Start with `RUN_MODE = "PILOT"`, then **Run all**. The environment cell may restart Colab once;
   after reconnection choose **Run all** again.
3. PILOT uses all 15 geometries, both generators, the original phase design, and 3 backgrounds
   per generator. It is non-reportable. Inspect response rates, traces, diagnostics, recovery and PPC.
4. FULL is unlocked only by a matching PILOT PASS. Change `RUN_MODE` to `"FULL"`, reconnect to a
   clean runtime and run all. FULL uses 100 backgrounds per generator and prior/link sensitivity.

Only A is collected and fitted. The overlap/patch coefficients are baseline covariates;
this notebook does not turn them into an M1 mitigation verdict. There is no LOO comparison here.
It does not need another notebook to be executed first. Compatible localisation arm data can be
reused from Drive after provenance checks; v3 quality-difference pair tables are incompatible.

### Frozen measurement and model

- `TONE_SNR=1.25`, `fs=512 Hz`, 480 context samples and 64 forecast samples.
- One lock and **two** adaptive non-lock controls, sharing background and phase, as in localisation.
- Raw median forecast, mean removal, rectangular window, 8192-point FFT, top-3 local maxima
  separated by at least 8 Hz. A hit means any retained peak lies within **1 Hz** of the injected tone.
- Every arm enters the fit. Truth and background-only hit rates are instrument diagnostics;
  a truth miss is not treated as proof that a forecast cannot hit.

\[
h_i\sim\mathrm{Bernoulli}(p_i),\quad
\mathrm{logit}(p_i)=\beta_{c[i]}+\gamma L_i+u_{k[i]}+u_{b[i]},
\]
\[
\beta_c\sim N(\bar\beta+\delta_O\widetilde O_c+\delta_P\widetilde{\log P}_c,\tau).
\]

`gamma` is the conditional log-odds ratio at a lock. Support means posterior probability at least
0.95 of an odds ratio below 0.8. Practical equivalence means probability at least 0.95 of
`abs(log odds ratio) < log(1.1)`. These concern **odds**, not amplitude or a relative hit-rate change.

### Exact computational reduction

Trials sharing every predictor have the same probability. Their Bernoulli likelihood can therefore
be evaluated as a Binomial count likelihood: `hits ~ Binomial(n_trials, p)`. The two log likelihoods
differ only by data-dependent combinatorial constants, so they give **the same parameter posterior**.
Both encodings are available in the model factory, and a numerical equivalence check runs before fitting.
Counts preserve unequal numbers of trials. This reduces the FULL likelihood from 1,113,000 arms to
82,000 groups for the current registry without dropping observations. No per-observation posterior
probabilities or log-likelihood matrices are stored.

The model still assumes conditional independence of arms given its group effects. A successful
synthetic test does not guarantee convergence or adequacy on Chronos data. No results are prefilled.


## 1. Setup
### 1.1 Locate the frozen repository
The notebook carries its analysis code. The repository supplies the frozen generators, geometry rules and checkpoint loader.


In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path
REPO_URL = "https://github.com/FedericoSabbadini/patchAliasing.git"
REPO_REVISION = "9d478e9a5aada82d138a47262ab5c2dc592911bc"
MARKER = Path("chronos/bayesian/probe_lib.py")
def on_colab():
    try:
        return importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        return False
IS_COLAB = on_colab()
def find_repo():
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / MARKER).is_file():
            return candidate
    target = Path("/content/patchAliasing") if IS_COLAB else here / "patchAliasing"
    if not (target / MARKER).is_file():
        subprocess.check_call(["git", "clone", "--no-checkout", REPO_URL, str(target)])
        subprocess.check_call(["git", "-C", str(target), "checkout", "--detach", REPO_REVISION])
    return target
REPO = find_repo()
BAYES_DIR = REPO / "chronos/bayesian"
sys.path.insert(0, str(BAYES_DIR)) if str(BAYES_DIR) not in sys.path else None
print("Repository:", REPO)


### 1.2 Install and verify the environment
On Colab this installs the repository lock and restarts once to avoid mixed binary packages. Local execution uses the active environment. No background or model is generated here.


In [ ]:
import json
import shutil
import sysconfig
import tempfile
import time

RESTART_STATE_PATH = Path(tempfile.gettempdir()) / "patchaliasing_A_bernoulli_env_state.json"
MAX_RESTARTS = 2


def uv_executable() -> str:
    found = shutil.which("uv")
    if found:
        return found
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])
    candidates = [
        Path(sysconfig.get_path("scripts")) / ("uv.exe" if os.name == "nt" else "uv"),
        Path(sys.executable).parent / ("uv.exe" if os.name == "nt" else "uv"),
    ]
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate)
    raise FileNotFoundError("uv was installed but its executable was not found")


def load_restart_state() -> dict:
    if RESTART_STATE_PATH.is_file():
        try:
            return json.loads(RESTART_STATE_PATH.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            pass
    return {"restarts": 0}


def save_restart_state(state: dict) -> None:
    RESTART_STATE_PATH.write_text(json.dumps(state), encoding="utf-8")


def clean_imports_are_healthy() -> bool:
    probe = subprocess.run(
        [
            sys.executable,
            "-c",
            "import numpy, scipy, pandas, pyarrow, arviz, pymc, nutpie, h5netcdf",
        ],
        capture_output=True,
        text=True,
    )
    if probe.returncode != 0:
        print(probe.stderr[-2500:])
    return probe.returncode == 0


def restart_colab(reason: str) -> None:
    print("=" * 78)
    print(f"RESTARTING THE COLAB RUNTIME: {reason}")
    print("This is deliberate. After reconnection choose Runtime > Run all again.")
    print("=" * 78)
    sys.stdout.flush()
    time.sleep(2)
    os.kill(os.getpid(), 9)


if not IS_COLAB:
    print("Local runtime: dependency installation skipped; using the active environment.")
else:
    state = load_restart_state()
    UV = uv_executable()

    if state["restarts"] == 0:
        with tempfile.TemporaryDirectory() as temporary:
            requirements = Path(temporary) / "requirements.locked.txt"
            subprocess.check_call(
                [
                    UV,
                    "export",
                    "--frozen",
                    "--no-dev",
                    "--no-emit-project",
                    "--no-hashes",
                    "--output-file",
                    str(requirements),
                ],
                cwd=REPO,
            )
            subprocess.check_call(
                [UV, "pip", "install", "--python", sys.executable, "--requirement", str(requirements)]
            )

        # Required for saving the small posterior checkpoint on Python 3.12+.
        subprocess.check_call(
            [UV, "pip", "install", "--python", sys.executable, "h5netcdf==1.8.1", "h5py==3.16.0"]
        )

        # Colab's preinstalled vision wheels can be ABI-incompatible with the locked torch.
        # They are not used by this analysis.
        subprocess.run(
            [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"],
            check=False,
            capture_output=True,
        )
        save_restart_state({"restarts": 1})
        restart_colab("locked environment installed")

    elif state["restarts"] == 1:
        if clean_imports_are_healthy():
            save_restart_state({"restarts": "verified"})
            print("Locked environment verified in the restarted runtime.")
        else:
            with tempfile.TemporaryDirectory() as temporary:
                requirements = Path(temporary) / "requirements.locked.txt"
                subprocess.check_call([UV, "export", "--frozen", "--no-dev", "--no-emit-project",
                                       "--no-hashes", "--output-file", str(requirements)], cwd=REPO)
                subprocess.check_call([UV, "pip", "install", "--python", sys.executable,
                                       "--reinstall-package", "numpy", "--reinstall-package", "scipy",
                                       "--requirement", str(requirements)])
            save_restart_state({"restarts": 2})
            restart_colab("NumPy/SciPy clean reinstall")
    else:
        if not clean_imports_are_healthy():
            raise RuntimeError(
                "The scientific imports are still broken after two restarts. Choose Runtime > "
                "Disconnect and delete runtime, reconnect to a fresh VM, and run all again."
            )
        save_restart_state({"restarts": "verified"})
        print("Locked environment verified.")


### 1.3 Settings
Change only PILOT to FULL after a matching PASS. The source path is optional; AUTO looks for the existing localisation collection, never for v3 pairs. Defaults and all analysis functions are fingerprinted; editing the model requires a new run namespace.


In [ ]:
import gc, hashlib, inspect, json, math, platform, time
from importlib import metadata as importlib_metadata
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
from scipy.special import expit, log_ndtr, gammaln
from IPython.display import display
import checkpointing as cp
import probe_lib as pl
import model_loader as ml

RUN_MODE = "PILOT"
NOTEBOOK_VERSION = "A-localisation-standalone-v1"
MODEL_VERSION = "A-bernoulli-centred-zerosum-exact-counts-v1"
RUN_ID = "model_A_localisation_bernoulli_v1"
SEED = 42
TONE_SNR, TOP_K, TOL_HZ, NFFT = 1.25, 3, 1.0, 8192
N_PHASE = 10
MODELS = list(pl.DELIVERABLE3_MODELS)
GENERATORS = tuple(pl.GENERATORS)
PRIOR_SCALE, BASELINE_SCALE, NU = 0.5, 1.5, 4
PRIOR_SCALES = (0.25, 0.5, 1.0)
SUPPORT_LOG_OR, ROPE_LOG_OR, PROB_CUTOFF = float(np.log(.8)), float(np.log(1.1)), .95
RHAT_MAX, PILOT_ESS_MIN, FULL_ESS_MIN = 1.01, 400, 1000
PPC_MIN_COVERAGE, SENSITIVITY_MAX_SPREAD = .90, .10
if RUN_MODE not in {"PILOT", "FULL"}:
    raise ValueError("RUN_MODE must be PILOT or FULL")
IS_FULL = RUN_MODE == "FULL"
N_BG = 100 if IS_FULL else 3
DRAWS = TUNE = 2000 if IS_FULL else 1000
CHAINS = 4
CORES = max(1, min(CHAINS, os.cpu_count() or 1))
TARGET_ACCEPT = .95
RECOVERY_DRAWS, RECOVERY_TUNE = 2000, 1500
PPC_DRAWS = 400
BATCH_SIZE, BG_PER_SHARD, SPECTRAL_BATCH = 64, 10, 256
NUTS_BACKEND = "nutpie" if importlib.util.find_spec("nutpie") else "pymc"
PAIR_REUSE = "AUTO"  # OFF collects fresh arms in this run.
SOURCE_DATA_OVERRIDE = os.environ.get("A_BERNOULLI_SOURCE_DIR", "")
if IS_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
DEFAULT_ROOT = Path("/content/drive/MyDrive/patchAliasing") if IS_COLAB else BAYES_DIR / "_run"
DRIVE_ROOT = Path(os.environ.get("A_BERNOULLI_DRIVE_ROOT", str(DEFAULT_ROOT)))
OUTPUT_ROOT = DRIVE_ROOT / ("full" if IS_FULL else "pilots") / RUN_ID
PILOT_RESULT_PATH = DRIVE_ROOT / "pilots" / RUN_ID / "final_verdict.json"
DATA_ROOT, CHECKPOINT_ROOT, FIGURE_ROOT = (OUTPUT_ROOT / x for x in ("data", "checkpoints", "figures"))
MANIFEST_PATH = OUTPUT_ROOT / "analysis_manifest.json"
for directory in (OUTPUT_ROOT, DATA_ROOT, CHECKPOINT_ROOT, FIGURE_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
pl.TONE_SNR = TONE_SNR
np.random.seed(SEED)
style = next((x for x in ("arviz-whitegrid", "seaborn-v0_8-whitegrid") if x in plt.style.available), "default")
plt.style.use(style)
print("Mode:", RUN_MODE, "| MCMC:", NUTS_BACKEND, "CPU cores:", CORES)
print("Backgrounds per generator:", N_BG, "| draws/tune/chains:", DRAWS, TUNE, CHAINS)
print("Output:", OUTPUT_ROOT)


## 2. Definitions
These cells define the complete A-only pipeline before any costly work. They also allow the manifest to fingerprint the executing functions instead of an unrelated repository copy of the notebook.


In [ ]:
def frequency_design():
    rows = []
    for P, S in MODELS:
        for f_lock in pl.f_lock(P, S):
            delta = pl.control_offset(P, S, f_lock)
            if not np.isfinite(delta):
                continue
            phases = pl.phases_Sf(f_lock, N_PHASE)
            for phase_idx, phase in enumerate(phases):
                for role, f in (("lock", f_lock), ("lo", f_lock-delta), ("hi", f_lock+delta)):
                    rows.append(dict(model=pl.model_tag(P,S), P=P, S=S, overlap=(P-S)/P,
                                     f_lock=float(f_lock), delta=float(delta), phase_idx=phase_idx,
                                     phase=float(phase), role=role, f=float(f), is_lock=int(role=="lock")))
    return pd.DataFrame(rows)

def spectral_peaks(values):
    values = np.atleast_2d(np.asarray(values, dtype=float))
    if values.shape[1] != pl.PRED or not np.isfinite(values).all():
        raise ValueError("Expected finite forecast horizons only")
    pieces = []
    for start in range(0, len(values), SPECTRAL_BATCH):
        block = values[start:start+SPECTRAL_BATCH]
        peaks = pl.dominant_freqs(block, k=TOP_K, nfft=NFFT, band=pl.BAND)
        # A truly flat forecast has no spectral peak. Avoid argmax assigning the first band bin.
        peaks[np.ptp(block, axis=1) == 0] = np.nan
        pieces.append(peaks)
    return np.concatenate(pieces)

def hit_values(peaks, frequencies):
    return pl.localisation_hit(peaks, frequencies, tol=TOL_HZ)

def package_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None

def logic_fingerprint(functions):
    # Code-object content fingerprints the executing functions without relying on notebook paths.
    # Exclude filenames/line numbers, which change between otherwise identical Colab executions.
    import types
    def normalise(value):
        if isinstance(value, types.CodeType):
            return dict(code=value.co_code.hex(), names=value.co_names, variables=value.co_varnames,
                        constants=[normalise(x) for x in value.co_consts],
                        argcount=value.co_argcount, kwonly=value.co_kwonlyargcount,
                        freevars=value.co_freevars, cellvars=value.co_cellvars)
        if isinstance(value, (tuple, list)):
            return [normalise(x) for x in value]
        if isinstance(value, (set, frozenset)):
            # Set display order depends on PYTHONHASHSEED; a clean kernel must keep the same hash.
            return {"set":sorted((normalise(x) for x in value), key=cp.canonical_json)}
        if isinstance(value, (str, int, float, bool)) or value is None:
            return value
        return repr(value)
    return cp.fingerprint({f.__name__:normalise(f.__code__) for f in functions})

def read_manifest():
    current = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    if current.get("analysis_fingerprint") != ANALYSIS_FINGERPRINT:
        raise ValueError("Run manifest changed; do not combine different analyses")
    return current

def record_artifact(path):
    path = Path(path)
    manifest = read_manifest()
    manifest["artifacts"][path.relative_to(OUTPUT_ROOT).as_posix()] = {
        "sha256":cp.sha256_file(path), "bytes":path.stat().st_size}
    cp.atomic_json(MANIFEST_PATH, manifest)

def valid_artifact(path):
    path = Path(path)
    entry = read_manifest()["artifacts"].get(path.relative_to(OUTPUT_ROOT).as_posix())
    if path.is_file() != (entry is not None):
        raise ValueError(f"Untracked or missing artifact: {path}")
    if entry is None:
        return False
    if cp.sha256_file(path) != entry["sha256"]:
        raise ValueError(f"Artifact hash mismatch: {path}")
    return True

def save_table(path, frame):
    cp.atomic_parquet(path, frame)
    record_artifact(path)

def canonical_background(generator, bg_id):
    path = DATA_ROOT / "backgrounds" / f"{generator}_{bg_id:03d}.npy"
    if valid_artifact(path):
        values = np.load(path)
    else:
        values = pl.background(generator, pl.CANON_LEN, 10000+bg_id)
        cp.atomic_npy(path, values)
        record_artifact(path)
    if values.shape != (pl.CANON_LEN,) or not np.isfinite(values).all():
        raise ValueError(f"Invalid canonical background: {path}")
    if abs(float(values.mean())) > 1e-5 or abs(float(values.std())-1) > 1e-5:
        raise ValueError(f"Background is not centred/unit variance: {path}")
    return values

def resolve_source():
    if PAIR_REUSE not in {"AUTO", "OFF"}:
        raise ValueError("PAIR_REUSE must be AUTO or OFF")
    if PAIR_REUSE == "OFF":
        return None
    root = Path(SOURCE_DATA_OVERRIDE) if SOURCE_DATA_OVERRIDE else DRIVE_ROOT / "full/d3_localisation_k3_v1/data"
    path = root / "02_contrasts.parquet"
    if not path.is_file():
        if SOURCE_DATA_OVERRIDE:
            raise FileNotFoundError(path)
        return None
    collection = json.loads((root / "collection_manifest.json").read_text(encoding="utf-8"))
    parent = json.loads((root.parent / "analysis_manifest.json").read_text(encoding="utf-8"))
    spec = parent["analysis_spec"]
    source_design = collection["design"]
    if cp.fingerprint(source_design) != collection.get("design_fingerprint"):
        raise ValueError("Source collection fingerprint mismatch")
    for name, expected in (("tone_snr", TONE_SNR), ("fhat_topk", TOP_K), ("fhat_tol_hz", TOL_HZ)):
        if spec.get(name) != expected:
            raise ValueError(f"Incompatible source {name}; use compatible data or PAIR_REUSE='OFF'")
    cfg = source_design["config"]
    if (cfg.get("n_phase_contrast") != N_PHASE or cfg.get("fhat_topk") != TOP_K
            or cfg.get("fhat_tol_hz") != TOL_HZ or tuple(cfg.get("generators", [])) != GENERATORS
            or cfg.get("n_bg", 0) < N_BG or cfg.get("smoke")):
        raise ValueError("Source collection does not cover this measurement/design")
    if source_design.get("checkpoints") != CHECKPOINT_IDENTITIES:
        raise ValueError("Source Chronos checkpoints differ")
    source_hashes = {name.replace("\\", "/"):digest for name,digest in source_design.get("source_sha256", {}).items()}
    for relative in ("chronos/bayesian/probe_lib.py", "chronos/bayesian/model_loader.py",
                     "chronos/data/synthetic/generators/kernelsynth_generator.py"):
        if source_hashes.get(relative) != cp.sha256_file(REPO/relative):
            raise ValueError(f"Source helper differs: {relative}")
    # The original collection manifest records NumPy, but not SciPy.
    for package in ("numpy",):
        if source_design.get("package_versions", {}).get(package) != package_version(package):
            raise ValueError(f"Source generator environment differs: {package}; use PAIR_REUSE='OFF'")
    entry = collection.get("merged", {}).get("contrasts", {})
    digest = cp.sha256_file(path)
    if entry.get("sha256") != digest:
        raise ValueError("Source arm-table hash mismatch")
    return dict(path=str(path), sha256=digest, n_bg=cfg["n_bg"],
                collection_fingerprint=collection["design_fingerprint"], rows=entry["rows"])

def collect_A(design, source=None):
    result_path = DATA_ROOT / "A_arms.parquet"
    if valid_artifact(result_path):
        return pd.read_parquet(result_path)
    if source is not None:
        path = Path(source["path"])
        if cp.sha256_file(path) != source["sha256"]:
            raise ValueError("Source changed since preflight")
        frame = pd.read_parquet(path)
        if len(frame) != source["rows"]:
            raise ValueError("Source row count mismatch")
        frame = frame[frame["bg_id"] < N_BG].copy().reset_index(drop=True)
        # Background-only calibration is cheap and was not retained in the original arm table.
        blind = np.zeros(len(frame), dtype=np.int8)
        for (generator, bg_id), positions in frame.groupby(["generator", "bg_id"]).indices.items():
            bg = canonical_background(generator, int(bg_id))
            peaks = spectral_peaks(bg[pl.CTX:])
            blind[positions] = hit_values(np.repeat(peaks, len(positions), axis=0), frame.iloc[positions]["f"])
        frame["h_blind"] = blind
        print("Reused localisation arms:", len(frame))
    else:
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Chronos collection device:", device)
        parts, start_all = [], time.time()
        for number, (P, S) in enumerate(MODELS, 1):
            tag = pl.model_tag(P,S)
            base = design[design["model"] == tag].reset_index(drop=True)
            probe = None
            try:
                for generator in GENERATORS:
                    for first_bg in range(0, N_BG, BG_PER_SHARD):
                        path = DATA_ROOT / "raw" / f"{tag}_{generator}_{first_bg:03d}.parquet"
                        if valid_artifact(path):
                            parts.append(pd.read_parquet(path))
                            continue
                        if probe is None:
                            probe = pl.Probe(P,S,device=device,batch_size=BATCH_SIZE)
                            if probe.checkpoint_identity != CHECKPOINT_IDENTITIES[tag]:
                                raise ValueError("Loaded Chronos checkpoint identity differs")
                        block_parts = []
                        for bg_id in range(first_bg, min(first_bg+BG_PER_SHARD, N_BG)):
                            bg = canonical_background(generator, bg_id)
                            meta = base.copy()
                            meta["generator"], meta["bg_id"] = generator, bg_id
                            signals = np.stack([pl.build_context(bg, row.f, row.phase, pl.CANON_LEN)
                                                for row in base.itertuples(index=False)])
                            predicted = probe.forecast(signals[:, :pl.CTX])
                            peaks = spectral_peaks(predicted)
                            truth_peaks = spectral_peaks(signals[:, pl.CTX:])
                            blind_peaks = spectral_peaks(bg[pl.CTX:])
                            meta["h"] = hit_values(peaks, meta["f"])
                            meta["h_truth"] = hit_values(truth_peaks, meta["f"])
                            meta["h_blind"] = hit_values(np.repeat(blind_peaks, len(meta), axis=0), meta["f"])
                            for j in range(TOP_K):
                                meta[f"f_hat_{j+1}"] = peaks[:, j]
                                meta[f"truth_f_hat_{j+1}"] = truth_peaks[:, j]
                            block_parts.append(meta)
                        part = pd.concat(block_parts, ignore_index=True)
                        save_table(path, part)
                        parts.append(part)
                        print(f"{tag} {generator} bg {first_bg}:{min(first_bg+BG_PER_SHARD,N_BG)} saved; "
                              f"elapsed {(time.time()-start_all)/60:.1f} min")
            finally:
                if probe is not None:
                    probe.close()
                gc.collect()
            print(f"Geometry {number}/{len(MODELS)} complete")
        frame = pd.concat(parts, ignore_index=True)
    validate_arms(frame, design)
    save_table(result_path, frame)
    return frame

def validate_arms(frame, design):
    keys = ["model", "generator", "bg_id", "f_lock", "phase_idx", "role"]
    if len(frame) != len(design)*len(GENERATORS)*N_BG or frame[keys].duplicated().any():
        raise ValueError("Incomplete/duplicate A-only arm design")
    if set(frame["generator"]) != set(GENERATORS) or set(frame["bg_id"]) != set(range(N_BG)):
        raise ValueError("Wrong background population")
    merged = frame.merge(design, on=["model", "f_lock", "phase_idx", "role"],
                         suffixes=("", "_expected"), how="left", validate="many_to_one")
    for name in ("P", "S", "overlap", "phase", "f", "delta", "is_lock"):
        if not np.allclose(merged[name], merged[name+"_expected"], rtol=0, atol=1e-9):
            raise ValueError(f"Arm metadata differs from the frozen design: {name}")
    sizes = frame.groupby(["generator", "bg_id"]).size()
    if len(sizes) != len(GENERATORS)*N_BG or not sizes.eq(len(design)).all():
        raise ValueError("Unequal or missing arm coverage per background")
    for name in ("h", "h_truth", "h_blind", "is_lock"):
        if not frame[name].isin([0,1]).all():
            raise ValueError(f"{name} must be binary")

def aggregate_trials(frame):
    # These, and only these, determine eta in Model A. Phase and control side share eta.
    keys = ["model", "P", "S", "overlap", "generator", "bg_id", "f_lock", "is_lock"]
    result = frame.groupby(keys, observed=True, sort=True).agg(
        hits=("h", "sum"), n_trials=("h", "size"),
        truth_hits=("h_truth", "sum"), blind_hits=("h_blind", "sum")).reset_index()
    for name in ("hits", "n_trials", "truth_hits", "blind_hits"):
        result[name] = result[name].astype(np.int64)
    if result["n_trials"].sum() != len(frame) or result["hits"].sum() != frame["h"].sum():
        raise AssertionError("Aggregation changed trials or successes")
    return result


### 2.1 Model factory
The priors, centred configuration hierarchy and zero-sum harmonic/background hierarchy are taken from localisation. `encoding="counts"` evaluates the exact sufficient-statistic likelihood; `encoding="bernoulli"` accepts individual arms for validation. The optional probit sensitivity is assessed later on a common odds-ratio scale, never by comparing raw link coefficients.


In [ ]:
def _codes(series):
    """Integer codes plus the ordered level names, for PyMC `coords`."""
    codes, levels = pd.factorize(series)
    return np.asarray(codes), list(map(str, levels))


def _overlap_scaled(df: pd.DataFrame, cfg_levels: list[str]) -> np.ndarray:
    """Centred, scaled patch overlap O = (P-S)/P; one unit = 0.5 of overlap."""
    O = df.groupby("model")["overlap"].first().reindex(cfg_levels).to_numpy(float)
    return (O - O.mean()) / 0.5


def _logP_centred(df: pd.DataFrame, cfg_levels: list[str]) -> np.ndarray:
    """Centred log patch size.

    Deliverable 3, H1: "The overlap enters as a ratio and the patch size as log P, because the
    patch grid has spacing fs/P: equal steps in log P are then equal ratios of spacing." On a raw-P
    scale one coefficient would make 8->16 and 16->24 the same change, which the geometry does not.
    """
    Pv = df.groupby("model")["P"].first().reindex(cfg_levels).to_numpy(float)
    lp = np.log(Pv)
    return lp - lp.mean()


# --------------------------------------------------------------------------------------- #
def model_A_contrast(df: pd.DataFrame, scale: float = PRIOR_SCALE,
                     baseline_scale: float = BASELINE_SCALE, nu: int = NU,
                     link: str = "logit", config_level: str = "both",
                     encoding: str = "counts") -> pm.Model:
    """Model A from localisation: shared log-odds lock effect and baseline covariates.
    Binomial counts are an exact aggregation of conditionally identical Bernoulli trials.
    Coefficients of overlap and log patch size do not modify the lock effect gamma.
    """
    if link not in {"logit", "probit"} or encoding not in {"counts", "bernoulli"}:
        raise ValueError("Invalid link or encoding")
    if config_level not in {"both", "overlap", "patch", "none"}:
        raise ValueError("Invalid configuration covariates")
    cfg_c, cfg_l = _codes(df["model"])
    harm_c, harm_l = _codes(df["f_lock"].round(3).astype(str))
    bg_c, bg_l = _codes(df["generator"] + "#" + df["bg_id"].astype(str))
    O_t = _overlap_scaled(df, cfg_l)
    lP_t = _logP_centred(df, cfg_l)
    is_lock = df["is_lock"].to_numpy(float)
    raw_y = df["hits" if encoding == "counts" else "h"].to_numpy(float)
    raw_n = df["n_trials"].to_numpy(float) if encoding == "counts" else np.ones(len(raw_y))
    if not (np.isfinite(raw_y).all() and np.isfinite(raw_n).all()
            and np.equal(raw_y, np.floor(raw_y)).all() and np.equal(raw_n, np.floor(raw_n)).all()):
        raise ValueError("Responses and trial counts must be finite integers")
    y, trials = raw_y.astype(int), raw_n.astype(int)
    if np.any(trials < 1) or np.any(y < 0) or np.any(y > trials):
        raise ValueError("Invalid binomial counts")

    coords = {"config": cfg_l, "harmonic": harm_l, "background": bg_l, "obs": np.arange(len(y))}
    with pm.Model(coords=coords) as m:
        #, population level: the estimand of H1 -------------------------------------
        gamma = pm.StudentT("gamma", nu=nu, mu=0.0, sigma=scale)
        beta_bar = pm.StudentT("beta_bar", nu=nu, mu=0.0, sigma=baseline_scale)
        cfg_mean = beta_bar
        if config_level in ("both", "overlap"):
            delta_O = pm.StudentT("delta_O", nu=nu, mu=0.0, sigma=scale)
            cfg_mean = cfg_mean + delta_O * O_t
        if config_level in ("both", "patch"):
            delta_P = pm.StudentT("delta_P", nu=nu, mu=0.0, sigma=scale)
            cfg_mean = cfg_mean + delta_P * lP_t

        # Preserve the source's centred configuration hierarchy and zero-sum group offsets.
        # Constraints separate baseline and group means; convergence still needs checking.
        tau = pm.HalfStudentT("tau", nu=nu, sigma=scale)
        beta = pm.Normal("beta", mu=cfg_mean, sigma=tau, dims="config")

        sigma_h = pm.HalfStudentT("sigma_harm", nu=nu, sigma=scale)
        u_h = pm.ZeroSumNormal("u_harm", sigma=sigma_h, dims="harmonic")
        sigma_b = pm.HalfStudentT("sigma_bg", nu=nu, sigma=scale)
        u_b = pm.ZeroSumNormal("u_bg", sigma=sigma_b, dims="background")

        eta = beta[cfg_c] + gamma * is_lock + u_h[harm_c] + u_b[bg_c]
        if link == "probit":
            p = pm.math.clip(0.5 * (1.0 + pm.math.erf(eta / np.sqrt(2.0))), 1e-9, 1-1e-9)
            parameter = {"p": p}
        else:
            parameter = {"logit_p": eta}
        if encoding == "counts":
            pm.Binomial("hits", n=trials, observed=y, dims="obs", **parameter)
        else:
            pm.Bernoulli("h", observed=y, dims="obs", **parameter)
        if link == "logit":
            pm.Deterministic("odds_ratio", pm.math.exp(gamma))
    return m


# --------------------------------------------------------------------------------------- #


### 2.2 Sampling, diagnostics and effects
Four independent chains, visible progress, atomic netCDF checkpoints and no pointwise log likelihood. All scalar/group parameters enter the convergence gate; `gamma` is also displayed separately. Probit effects are converted into conditional log-odds differences at the same control-design rows, weighted by trial count.


In [ ]:
def fit_or_load(label, frame, scale=PRIOR_SCALE, link="logit", draws=None, tune=None):
    path = CHECKPOINT_ROOT / f"{label}.nc"
    if valid_artifact(path):
        idata = az.from_netcdf(path)
        idata.load()
        idata.close()
        return idata
    # Approximately match the width of the priors in probit units (logit/probit slope ~1.6).
    unit = 1.6 if link == "probit" else 1.0
    model = model_A_contrast(frame, scale=scale/unit, baseline_scale=BASELINE_SCALE/unit, link=link)
    start = time.time()
    print(f"{label}: {len(frame):,} groups / {frame.n_trials.sum():,} Bernoulli trials; "
          f"draws={draws or DRAWS}, tune={tune or TUNE}, chains={CHAINS}, cores={CORES}")
    kwargs = {"nuts_sampler":NUTS_BACKEND} if NUTS_BACKEND != "pymc" else {}
    with model:
        point = model.initial_point()
        if not np.isfinite(model.compile_logp()(point)):
            raise ValueError("Non-finite initial log probability")
        idata = pm.sample(draws=draws or DRAWS, tune=tune or TUNE, chains=CHAINS, cores=CORES,
                          random_seed=SEED, target_accept=TARGET_ACCEPT, progressbar=True,
                          idata_kwargs={"log_likelihood":False}, **kwargs)
    if "log_likelihood" in idata or "p" in idata.posterior:
        raise AssertionError("An observation-sized posterior array was unexpectedly retained")
    cp.atomic_netcdf(path, idata)
    record_artifact(path)
    print(f"Saved {path.name} in {(time.time()-start)/60:.1f} minutes")
    return idata

def diagnostics_for(idata, ess_min):
    table = az.summary(idata, kind="diagnostics", round_to="none")
    table = table[["r_hat", "ess_bulk", "ess_tail"]].apply(pd.to_numeric, errors="coerce")
    divergences = int(np.asarray(idata.sample_stats["diverging"]).sum())
    finite = np.isfinite(table.to_numpy(float)).all()
    result = dict(max_rhat=float(table.r_hat.max()), min_ess_bulk=float(table.ess_bulk.min()),
                  min_ess_tail=float(table.ess_tail.min()), divergences=divergences)
    result["passed"] = bool(finite and (table.r_hat < RHAT_MAX).all()
                            and (table.ess_bulk > ess_min).all() and (table.ess_tail > ess_min).all()
                            and divergences == 0)
    return result, table

def posterior_array(idata, name, dimension=None):
    dims = ["chain", "draw"] + ([dimension] if dimension else [])
    values = np.asarray(idata.posterior[name].transpose(*dims), dtype=float)
    return values.reshape(-1, values.shape[-1]) if dimension else values.ravel()

def posterior_components(idata, frame):
    post = idata.posterior
    def index(values, dimension):
        lookup = {str(v):i for i,v in enumerate(post.coords[dimension].values)}
        result = np.array([lookup.get(str(v), -1) for v in values], dtype=int)
        if np.any(result < 0):
            raise ValueError(f"Unknown posterior level: {dimension}")
        return result
    return dict(beta=posterior_array(idata,"beta","config"),
                harm=posterior_array(idata,"u_harm","harmonic"),
                bg=posterior_array(idata,"u_bg","background"), gamma=posterior_array(idata,"gamma"),
                ci=index(frame.model,"config"), hi=index(frame.f_lock.round(3).astype(str),"harmonic"),
                bi=index(frame.generator+"#"+frame.bg_id.astype(str),"background"))

def conditional_log_or_draws(idata, frame, link="logit"):
    if link == "logit":
        return posterior_array(idata,"gamma")
    # No observation-wise deterministic is put into the PyMC posterior.
    controls = frame[frame.is_lock.eq(0)].reset_index(drop=True)
    arrays = posterior_components(idata, controls)
    weights = controls.n_trials.to_numpy(float)
    weights /= weights.sum()
    results = []
    for i,gamma in enumerate(arrays["gamma"]):
        eta = arrays["beta"][i,arrays["ci"]]+arrays["harm"][i,arrays["hi"]]+arrays["bg"][i,arrays["bi"]]
        # Match the numerical probability bounds used in the model's probit likelihood.
        from scipy.special import ndtr, logit
        before = logit(np.clip(ndtr(eta),1e-9,1-1e-9))
        after = logit(np.clip(ndtr(eta+gamma),1e-9,1-1e-9))
        results.append(float(weights @ (after-before)))
    return np.asarray(results)

def effect_summary(draws):
    low, median, high = np.quantile(draws,[.025,.5,.975])
    return dict(median_log_or=float(median), eti_low_log_or=float(low), eti_high_log_or=float(high),
                median_odds_ratio=float(np.exp(median)),
                p_support=float(np.mean(draws < SUPPORT_LOG_OR)),
                p_equivalence=float(np.mean(np.abs(draws) < ROPE_LOG_OR)))

def predictive_check(idata, frame, link="logit", seed=SEED+600):
    arrays = posterior_components(idata, frame)
    rng = np.random.default_rng(seed)
    selected = np.linspace(0,len(arrays["gamma"])-1,min(PPC_DRAWS,len(arrays["gamma"]))).astype(int)
    strata = {}
    for columns in (("model","is_lock"),("generator","is_lock")):
        for levels,positions in frame.groupby(list(columns), observed=True).indices.items():
            strata[str(tuple(zip(columns,levels)))] = np.asarray(positions,int)
    samples = {name:[] for name in strata}
    n = frame.n_trials.to_numpy(int)
    lock = frame.is_lock.to_numpy(float)
    for i in selected:
        eta = (arrays["beta"][i,arrays["ci"]]+arrays["gamma"][i]*lock
               +arrays["harm"][i,arrays["hi"]]+arrays["bg"][i,arrays["bi"]])
        if link == "logit":
            p = expit(eta)
        else:
            from scipy.special import ndtr
            p = np.clip(ndtr(eta),1e-9,1-1e-9)
        simulated = rng.binomial(n,p)
        for name,positions in strata.items():
            samples[name].append(simulated[positions].sum()/n[positions].sum())
    rows = []
    for name,positions in strata.items():
        observed = frame.hits.to_numpy()[positions].sum()/n[positions].sum()
        low,high = np.quantile(samples[name],[.025,.975])
        rows.append(dict(stratum=name,n_trials=int(n[positions].sum()),observed_rate=float(observed),
                         rep_low=float(low),rep_high=float(high),passed=bool(low<=observed<=high)))
    return pd.DataFrame(rows)

def recovery_data(groups):
    rng = np.random.default_rng(SEED+300)
    out = groups[groups.bg_id < 3].copy().reset_index(drop=True)
    ci,cl = _codes(out.model)
    hi,hl = _codes(out.f_lock.round(3).astype(str))
    bi,bl = _codes(out.generator+"#"+out.bg_id.astype(str))
    truth = {"gamma":-.60,"beta_bar":-.40,"delta_O":.20,"delta_P":-.15}
    means = truth["beta_bar"]+truth["delta_O"]*_overlap_scaled(out,cl)+truth["delta_P"]*_logP_centred(out,cl)
    beta = rng.normal(means,.20)
    harm = rng.normal(0,.20,len(hl)); harm -= harm.mean()
    bg = rng.normal(0,.20,len(bl)); bg -= bg.mean()
    eta = beta[ci]+truth["gamma"]*out.is_lock.to_numpy()+harm[hi]+bg[bi]
    out["hits"] = rng.binomial(out.n_trials.to_numpy(int),expit(eta))
    return out,truth

def verdict_from(gate_ok, effect):
    if not gate_ok:
        return "NOT REPORTABLE"
    if effect["p_support"] >= PROB_CUTOFF:
        return "SUPPORTED"
    if effect["p_equivalence"] >= PROB_CUTOFF:
        return "PRACTICALLY EQUIVALENT"
    return "INCONCLUSIVE"


## 3. Preflight and immutable run manifest
This checks the estimator, counts actual forecast work, records data sources and separates PILOT/FULL artifacts. The instrument checks use isolated synthetic tones, not empirical results. No matrix of shape posterior draws × individual arms is required.


In [ ]:
design = frequency_design()
CHECKPOINT_IDENTITIES = {pl.model_tag(P,S):ml.checkpoint_identity(P,S) for P,S in MODELS}
test_f = np.array([32.,64.,96.])
test_signals = np.stack([pl.make_tone(f,.37,pl.PRED) for f in test_f])
assert hit_values(spectral_peaks(test_signals),test_f).all()
assert np.isnan(spectral_peaks(np.zeros((1,pl.PRED)))).all()
assert not hit_values(np.full((1,TOP_K),np.nan),[32.]).any()
SOURCE = resolve_source()
source_paths = [BAYES_DIR/x for x in ("probe_lib.py","model_loader.py","checkpointing.py")]
source_paths += sorted((REPO/"chronos/data/synthetic").rglob("*.py"))
helper_hashes = {path.relative_to(REPO).as_posix():cp.sha256_file(path) for path in source_paths}
analysis_functions = [value for name,value in list(globals().items())
                      if inspect.isfunction(value) and value.__module__ == "__main__"
                      and name not in {"display"}]
SCIENCE_SPEC = dict(notebook=NOTEBOOK_VERSION,model=MODEL_VERSION,models=MODELS,generators=GENERATORS,
                    snr=TONE_SNR,top_k=TOP_K,tolerance_hz=TOL_HZ,nfft=NFFT,n_phase=N_PHASE,seed=SEED,
                    prior_scale=PRIOR_SCALE,baseline_scale=BASELINE_SCALE,nu=NU,
                    prior_scales=PRIOR_SCALES,likelihood_encoding="exact_binomial_counts",
                    scope="all_arms",support_log_or=SUPPORT_LOG_OR,rope_log_or=ROPE_LOG_OR,
                    cutoff=PROB_CUTOFF,rhat_max=RHAT_MAX,ess_pilot=PILOT_ESS_MIN,ess_full=FULL_ESS_MIN,
                    ppc_min_coverage=PPC_MIN_COVERAGE,sensitivity_max_spread=SENSITIVITY_MAX_SPREAD,
                    design_hash=cp.fingerprint(design.to_dict("records")),
                    helper_hashes=helper_hashes,checkpoints=CHECKPOINT_IDENTITIES,
                    executing_logic_sha256=logic_fingerprint(analysis_functions))
CORE_FINGERPRINT = cp.fingerprint(SCIENCE_SPEC)
if IS_FULL:
    if not PILOT_RESULT_PATH.is_file():
        raise FileNotFoundError(f"Run a PILOT first: {PILOT_RESULT_PATH}")
    previous = json.loads(PILOT_RESULT_PATH.read_text(encoding="utf-8"))
    if previous.get("core_fingerprint") != CORE_FINGERPRINT or previous.get("pilot_route_ok") is not True:
        raise ValueError("FULL requires a matching PILOT PASS under this measurement and model")
ANALYSIS_SPEC = dict(science=SCIENCE_SPEC,core_fingerprint=CORE_FINGERPRINT,run_id=RUN_ID,run_mode=RUN_MODE,
                     n_bg=N_BG,draws=DRAWS,tune=TUNE,chains=CHAINS,target_accept=TARGET_ACCEPT,
                     recovery_draws=RECOVERY_DRAWS,recovery_tune=RECOVERY_TUNE,ppc_draws=PPC_DRAWS,
                     backend=NUTS_BACKEND,python=platform.python_version(),source=SOURCE,
                     packages={x:package_version(x) for x in ("numpy","pandas","pymc","arviz","nutpie","torch","chronos-forecasting")})
ANALYSIS_FINGERPRINT = cp.fingerprint(ANALYSIS_SPEC)
if MANIFEST_PATH.is_file():
    existing = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    if existing.get("analysis_fingerprint") != ANALYSIS_FINGERPRINT:
        raise ValueError("Existing run uses different code/settings/data. Choose a fresh RUN_ID; no auto-heal.")
else:
    cp.atomic_json(MANIFEST_PATH,dict(analysis_fingerprint=ANALYSIS_FINGERPRINT,analysis_spec=ANALYSIS_SPEC,artifacts={}))
work = design.groupby("model").size().rename("arms_per_background").to_frame()
work["forecasts"] = work.arms_per_background*N_BG*len(GENERATORS)
display(work)
print("Total A-only forecasts:",int(work.forecasts.sum()))
print("Source:",SOURCE["path"] if SOURCE else "new collection")
print("Avoided pointwise log-likelihood allocation:",round(int(work.forecasts.sum())*DRAWS*CHAINS*8/2**30,2),"GiB")
print("Instrument unit checks passed; empirical convergence has not been tested.")


## 4. Collect or reuse A arms
Only forecast calls are needed. One model is loaded at a time, backgrounds and completed shards are saved on Drive, and progress is printed per shard. Collection does not run MDL, collapse, other Bayesian models or parameter recovery for C.


In [ ]:
arms = collect_A(design,SOURCE)
validate_arms(arms,design)
groups = aggregate_trials(arms)
save_table(DATA_ROOT/"A_counts.parquet",groups)
print(f"{len(arms):,} individual Bernoulli trials -> {len(groups):,} exact count groups")
print("Both outcomes present:",sorted(arms.h.unique().tolist()))
if arms.h.nunique() < 2:
    raise ValueError("All forecasts have the same outcome; inspect the instrument before fitting")
rate_table = arms.groupby(["generator","role"]).agg(
    n=("h","size"),forecast_hit=("h","mean"),truth_hit=("h_truth","mean"),background_hit=("h_blind","mean")).reset_index()
rate_table["truth_minus_background"] = rate_table.truth_hit-rate_table.background_hit
display(rate_table)
save_table(DATA_ROOT/"instrument_rates.parquet",rate_table)
# Minimal calibration requirement: on each generator/role, the injected true future improves
# detection over the same background alone. This is checked before interpreting model effects.
MEASUREMENT_OK = bool((rate_table.truth_minus_background > 0).all())
print("Instrument separation gate:",MEASUREMENT_OK)
print("Truth and background rates can differ by role; inspect these differences before a scientific interpretation.")
display(arms.groupby(["model","role"]).h.agg(["size","mean"]))


### 4.1 Verify the likelihood reduction
Compare both model encodings at identical parameter values. Equality is checked after removing the Binomial combinatorial constant, which contains no parameter. This test includes unequal trial counts.


In [ ]:
small_arms = arms[(arms.model.isin(arms.model.unique()[:3])) & (arms.bg_id < 2)].copy()
small_counts = aggregate_trials(small_arms)
# Equal ordering of coordinates makes the comparison use the same latent parameter values.
small_arms = small_arms.sort_values(["model","P","S","overlap","generator","bg_id","f_lock","is_lock"]).reset_index(drop=True)
bernoulli_model = model_A_contrast(small_arms,encoding="bernoulli")
count_model = model_A_contrast(small_counts)
point = count_model.initial_point()
constant = np.sum(gammaln(small_counts.n_trials+1)-gammaln(small_counts.hits+1)
                  -gammaln(small_counts.n_trials-small_counts.hits+1))
bern_logp = bernoulli_model.compile_logp()
count_logp = count_model.compile_logp()
for gamma in (-.6,0.,.4):
    candidate = {key:np.array(value,copy=True) for key,value in point.items()}
    candidate["gamma"] = np.asarray(gamma)
    np.testing.assert_allclose(count_logp(candidate)-bern_logp(candidate),constant,rtol=1e-9,atol=1e-7)
print("Exact Bernoulli/Binomial posterior equivalence: PASS")
del small_arms,small_counts,bernoulli_model,count_model,bern_logp,count_logp
gc.collect()


## 5. Synthetic parameter recovery, A only
This simulates binomial successes under known A parameters on the same design axes and fits the same model. It tests implementation and identifiability, not H1. Both interval coverage and convergence are required. Its outputs are explicitly labelled synthetic and are stored separately from the empirical posterior.


In [ ]:
recovery_frame,recovery_truth = recovery_data(groups)
recovery_idata = fit_or_load("synthetic_parameter_recovery_A",recovery_frame,
                             draws=RECOVERY_DRAWS,tune=RECOVERY_TUNE)
ess_threshold = FULL_ESS_MIN if IS_FULL else PILOT_ESS_MIN
recovery_diagnostic,recovery_parameters = diagnostics_for(recovery_idata,ess_threshold)
recovery_rows = []
for name,truth in recovery_truth.items():
    low,median,high = np.quantile(posterior_array(recovery_idata,name),[.025,.5,.975])
    recovery_rows.append(dict(parameter=name,truth=truth,median=float(median),eti_low=float(low),
                              eti_high=float(high),covered=bool(low<=truth<=high)))
recovery_table = pd.DataFrame(recovery_rows)
RECOVERY_OK = bool(recovery_table.covered.all() and recovery_diagnostic["passed"])
save_table(OUTPUT_ROOT/"synthetic_recovery_summary.parquet",recovery_table)
display(recovery_table)
print("SYNTHETIC ONLY:",recovery_diagnostic,"Recovery gate:",RECOVERY_OK)
del recovery_idata,recovery_frame
gc.collect()


## 6. Primary Model A fit
The progress table reports elapsed time for the actual CPU fit. Resuming loads the verified completed posterior. Increasing the nominal number of draws does not establish convergence; the next cells check it.


In [ ]:
idata_A = fit_or_load("primary_A_logit",groups)
primary_diagnostic,parameter_diagnostics = diagnostics_for(idata_A,ess_threshold)
primary_effect = effect_summary(conditional_log_or_draws(idata_A,groups))
PRIMARY_OK = primary_diagnostic["passed"]
save_table(OUTPUT_ROOT/"primary_diagnostics.parquet",parameter_diagnostics.reset_index(names="parameter"))
display(pd.DataFrame([primary_effect]))
print("Primary convergence:",primary_diagnostic)
display(parameter_diagnostics.loc[["gamma","beta_bar","delta_O","delta_P","tau","sigma_harm","sigma_bg"]])
display(parameter_diagnostics.sort_values("r_hat",ascending=False).head(10))
display(parameter_diagnostics.sort_values("ess_bulk").head(10))


### 6.1 Traces and stratified predictive checks
The checks preserve trial-count weights and include model × lock/control and generator × lock/control strata. Replicates are generated one draw at a time. They do not allocate a draws × observations array.


In [ ]:
trace_names = ["gamma","beta_bar","delta_O","delta_P","tau","sigma_harm","sigma_bg"]
fig,axes = plt.subplots(len(trace_names),1,figsize=(11,13),sharex=True)
for axis,name in zip(axes,trace_names):
    values = np.asarray(idata_A.posterior[name].transpose("chain","draw"))
    for chain,draws in enumerate(values):
        axis.plot(draws,lw=.6,label=f"chain {chain}")
    axis.set_ylabel(name)
axes[0].legend(ncol=CHAINS,fontsize=8)
axes[-1].set_xlabel("retained draw")
fig.suptitle(f"Model A Bernoulli, {RUN_MODE}")
fig.tight_layout()
fig.savefig(FIGURE_ROOT/"primary_traces.png",dpi=130)
plt.show()
ppc_table = predictive_check(idata_A,groups)
PPC_OK = bool(ppc_table.passed.mean() >= PPC_MIN_COVERAGE)
save_table(OUTPUT_ROOT/"primary_ppc.parquet",ppc_table)
display(ppc_table)
print("PPC coverage:",ppc_table.passed.mean(),"gate:",PPC_OK)


## 7. FULL prior and link sensitivity
FULL fits prior scales 0.25 and 1.0 in addition to the primary 0.5, plus a probit link with approximately matched prior widths. All comparisons use the conditional log-odds ratio averaged over the same weighted control design. Both support and equivalence probabilities must vary by at most 0.10. The probit coefficient itself is never interpreted as a log-odds ratio.


In [ ]:
sensitivity_rows = []
if IS_FULL and PRIMARY_OK:
    sensitivity_rows.append(dict(variant="primary logit .5",passed=PRIMARY_OK,**primary_effect))
    for label,scale,link in (("logit .25",.25,"logit"),("logit 1.0",1.,"logit"),("probit matched .5",.5,"probit")):
        fit = fit_or_load("sensitivity_"+label.replace(" ","_"),groups,scale=scale,link=link)
        diagnostic,_ = diagnostics_for(fit,FULL_ESS_MIN)
        effect = effect_summary(conditional_log_or_draws(fit,groups,link=link))
        sensitivity_rows.append(dict(variant=label,passed=diagnostic["passed"],**effect))
        del fit
        gc.collect()
    sensitivity_table = pd.DataFrame(sensitivity_rows)
    spread = sensitivity_table[["p_support","p_equivalence"]].max()-sensitivity_table[["p_support","p_equivalence"]].min()
    SENSITIVITY_OK = bool(sensitivity_table.passed.all() and (spread <= SENSITIVITY_MAX_SPREAD).all())
    save_table(OUTPUT_ROOT/"sensitivity.parquet",sensitivity_table)
    display(sensitivity_table)
    print("Probability spreads:",spread.to_dict())
else:
    sensitivity_table = pd.DataFrame()
    SENSITIVITY_OK = False
    print("Sensitivity deferred: PILOT mode or failed primary convergence.")


## 8. Final A-only result
PILOT can only license a larger run. FULL produces an H1 result only if the instrument, synthetic recovery, primary convergence, predictive and sensitivity gates all pass. Failed gates are never interpreted as evidence against H1. The scope remains the full, unconditioned arm population.


In [ ]:
PILOT_ROUTE_OK = bool(MEASUREMENT_OK and RECOVERY_OK and PRIMARY_OK and PPC_OK)
FINAL_GATE_OK = bool(IS_FULL and PILOT_ROUTE_OK and SENSITIVITY_OK)
final_verdict = verdict_from(FINAL_GATE_OK,primary_effect)
gate_table = pd.DataFrame([
    {"gate":"instrument separation","passed":MEASUREMENT_OK},
    {"gate":"A synthetic recovery","passed":RECOVERY_OK},
    {"gate":"A convergence","passed":PRIMARY_OK},
    {"gate":"A PPC","passed":PPC_OK},
    {"gate":"FULL mode","passed":IS_FULL},
    {"gate":"prior/link sensitivity","passed":SENSITIVITY_OK}])
result = dict(run_mode=RUN_MODE,model_version=MODEL_VERSION,core_fingerprint=CORE_FINGERPRINT,
              analysis_fingerprint=ANALYSIS_FINGERPRINT,pilot_route_ok=PILOT_ROUTE_OK,
              gate_ok=FINAL_GATE_OK,verdict=final_verdict,scope="all_arms",effect=primary_effect,
              primary_diagnostic=primary_diagnostic,gates=gate_table.to_dict("records"))
cp.atomic_json(OUTPUT_ROOT/"final_verdict.json",result)
record_artifact(OUTPUT_ROOT/"final_verdict.json")
save_table(OUTPUT_ROOT/"gates.parquet",gate_table)
display(gate_table)
print("FINAL H1:",final_verdict)
if not IS_FULL:
    print("PILOT PASS: change only RUN_MODE to FULL and use a clean runtime." if PILOT_ROUTE_OK
          else "PILOT FAIL: inspect rates, recovery, traces and failed gates before FULL.")
print("Artifacts:",OUTPUT_ROOT)


## Interpretation and limits

A positive `delta_O` changes the baseline hit odds of both controls and locks in this model; it does not demonstrate mitigation of the lock-specific effect. No M1 verdict is issued. The scientific result concerns inclusion of the target among the top-3 forecast peaks, not its amplitude and not the incremental response used by v3. Calibration rates and posterior uncertainty are conditional on the chosen signal generators, SNR, estimator and one trained checkpoint per geometry.

All displayed empirical conclusions must come from your execution. The delivered notebook has no stored fit output. For local execution use the project environment and run this file top-to-bottom; in Colab use the instructions at the top. FULL can be costly even after exact aggregation, so use the measured pilot progress and diagnostics before budgeting it.
